# Notebook 17b — NSV Data Preparation on Lowpassed Data (BSISO MJJAS, lp25)
**Project:** ENSO-BSISO SSL — Neural State Variables extension  
**Author:** Jiayi (jh9141@nyu.edu)

Variant of `nb17` that reads from the **25-day lowpassed** BSISO data instead of raw Lee preprocessing. Same pair-construction logic, different source.

## Why the lp25 variant

nb19 (run on nb18's Lee-only latents) returned `d̂ ≈ 17` with a failed noise-calibration control and a flat PCA scree. Diagnosis (see chat): the Stage 1 encoder absorbed **synoptic-eddy noise** (the ~5–10 day weather variability that dominates daily anomaly variance), not just the BSISO state. The result is a high-dimensional latent that the Levina-Bickel estimator can't measure reliably.

The fix: feed the encoder **lowpassed** input fields where signals faster than 25 days have been removed. The BSISO band (30–60 d) survives; the synoptic noise is gone. This is the same `X_MJJAS_lee_lp25.npy` file `nb08` used for its SSL training — proven to produce a clean 2-D circular embedding (z = 14.55 ENSO displacement).

## Source files

| File | What it is |
|---|---|
| `X_MJJAS_lee_lp25.npy` | Lee preprocessing + Lanczos 25-day lowpass applied per year's MJJAS block. Shape `(N, 3, 31, 51)`. Same domain, channels `[u850, v850, OLR]`. |
| `labels_aligned_mjjas_lee_lp25.csv` | Labels aligned to the lp25 series (25 edge days dropped from each end of each year's MJJAS block by the filter). |

## What's the same / what's different vs nb17

| | nb17 (Lee only) | nb17b (lp25) |
|---|---|---|
| Source X | `X_MJJAS_lee.npy` | `X_MJJAS_lee_lp25.npy` |
| Source labels | `labels_aligned_mjjas_lee_clean.csv` | `labels_aligned_mjjas_lee_lp25.csv` |
| Pair rule | `delta=1d AND same year` | same |
| Train/val split | every 5th year | same |
| Output Drive folder | `nsv/data/` | **`nsv/data_lp25/`** |
| Days per year after preprocessing | 153 | ~103 (25 dropped from each MJJAS edge) |
| **Expected pair count** | 6,536 | **~4,386** (43 yrs × 102 pairs/yr) |
| What's in the fields | BSISO + synoptic + everything | BSISO + slow modes, synoptic removed |

## Outputs (`BSISO_SSL_Project/nsv/data_lp25/`)

All filenames identical to nb17, just in a separate folder so the Lee-only baseline stays on disk for comparison.

## Verification gate (must pass before nb18b)
1. **No cross-year pairs.** Assert `max(dates_t1 - dates_t) == 1 day`.
2. **Pair count in range** `[43 × 95, 43 × 105]` (i.e., ~4,085 to ~4,515) — accounts for lp25's per-year edge-drop.
3. **ENSO balance.** Train and val splits both contain EN, Neutral, and LN.
4. **Visual coherence.** Three random `(X_t, X_t1)` pairs as OLR maps should look **smoother** than nb17's plot (synoptic eddies removed).

## Runtime
~2 min on Colab T4.

## N=4,386 is still healthy for Levina-Bickel

`log₂(4386) ≈ 12`, so the ID estimator stays reliable up to ~d=10. nb19's `k_list = int(N × {0.008..0.016}) = [35, 43, 52, 61, 70]` — comfortable neighborhood sizes. If the true ID is in the 2–5 range we expect for BSISO, this sample size is more than adequate.

---

## Cell 1 — Mount Drive + Constants

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR  = f'{PROJECT_DIR}/data/processed'
NSV_DIR        = f'{PROJECT_DIR}/nsv'
NSV_DATA_DIR   = f'{NSV_DIR}/data_lp25'                    # NEW: lp25 lives in a separate folder
os.makedirs(NSV_DATA_DIR, exist_ok=True)

X_FILE      = 'X_MJJAS_lee_lp25.npy'                       # 25-day lowpassed (same file as nb08)
LABELS_FILE = 'labels_aligned_mjjas_lee_lp25.csv'          # labels aligned to lp25 series

VAL_YEARS_STRIDE = 5

print(f'Project dir:  {PROJECT_DIR}')
print(f'Source X:     {PROCESSED_DIR}/{X_FILE}')
print(f'Labels:       {PROCESSED_DIR}/{LABELS_FILE}')
print(f'Output dir:   {NSV_DATA_DIR}')

## Cell 2 — Load Data, Build Consecutive Pairs, Year-Based Split, Save

Identical logic to nb17 Cell 2 — just different source files and different output folder. The pair-construction expression `(delta_days == 1) & same_year` correctly handles the bandpass edge-drops by construction (drops just shorten the within-year sequence, never invalidate adjacency).

In [ ]:
X = np.load(f'{PROCESSED_DIR}/{X_FILE}')
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])
labels['date'] = labels['date'].dt.normalize()

assert X.shape[0] == len(labels), f'X / labels length mismatch: {X.shape[0]} vs {len(labels)}'
assert X.shape[1:] == (3, 31, 51), f'Unexpected X shape: {X.shape[1:]}; expected (3, 31, 51).'
print(f'Raw load:  X shape {X.shape}, {len(labels)} labels.  Date range {labels["date"].min().date()} to {labels["date"].max().date()}.')

# Defensive sort + reset (should be no-op if nb08 wrote in order)
order = np.argsort(labels['date'].values)
if not np.array_equal(order, np.arange(len(labels))):
    print(f'WARN: labels not pre-sorted — reordering X and labels.')
    X = X[order]
    labels = labels.iloc[order].reset_index(drop=True)

dates_all = pd.DatetimeIndex(labels['date'].values)
years_all = dates_all.year.values

# Build valid-pair-start mask. valid[i]=True means (X[i], X[i+1]) is a valid pair.
delta_days = (dates_all[1:] - dates_all[:-1]).days
same_year  = years_all[1:] == years_all[:-1]
valid_start = (delta_days == 1) & same_year
valid_start = np.concatenate([valid_start, [False]])

pair_idx_t  = np.where(valid_start)[0]
pair_idx_t1 = pair_idx_t + 1
N_pairs = len(pair_idx_t)
print(f'\nPair construction:  {N_pairs} valid consecutive-day pairs (compare nb17: 6,536; lp25 expected slightly less due to bandpass edge drop).')

X_t  = X[pair_idx_t].astype(np.float32)
X_t1 = X[pair_idx_t1].astype(np.float32)
dates_t          = dates_all[pair_idx_t]
bsiso_phase_t    = labels['bsiso_phase'].values[pair_idx_t].astype(np.int8)
bsiso_amplitude_t = labels['bsiso_amplitude'].values[pair_idx_t].astype(np.float32)
enso_cat_t       = labels['enso_category'].values[pair_idx_t].astype('<U10')

pair_years  = dates_t.year.values
all_years   = sorted(np.unique(years_all).tolist())
val_years   = all_years[::VAL_YEARS_STRIDE]
train_years = sorted(set(all_years) - set(val_years))
train_mask  = np.isin(pair_years, train_years)

n_train = int(train_mask.sum())
n_val   = int((~train_mask).sum())
print(f'Train: {n_train} pairs ({100*n_train/N_pairs:.1f}%)  from {len(train_years)} years')
print(f'Val:   {n_val} pairs ({100*n_val/N_pairs:.1f}%)  from {len(val_years)} years: {val_years}')

np.save(f'{NSV_DATA_DIR}/X_t.npy', X_t)
np.save(f'{NSV_DATA_DIR}/X_t1.npy', X_t1)
np.save(f'{NSV_DATA_DIR}/dates_t.npy', dates_t.values.astype('datetime64[D]'))
np.save(f'{NSV_DATA_DIR}/bsiso_phase_t.npy', bsiso_phase_t)
np.save(f'{NSV_DATA_DIR}/bsiso_amplitude_t.npy', bsiso_amplitude_t)
np.save(f'{NSV_DATA_DIR}/enso_cat_t.npy', enso_cat_t)
np.save(f'{NSV_DATA_DIR}/train_mask.npy', train_mask)

meta = {
    'source_X':         X_FILE,
    'source_labels':    LABELS_FILE,
    'preprocessing':    'Lee MJJAS + Lanczos 25-day lowpass (synoptic noise removed)',
    'channels':         ['u850', 'v850', 'OLR'],
    'spatial_shape':    list(X.shape[1:]),
    'temporal_scope':   'MJJAS (May 1 – September 30), lp25',
    'pair_rule':        'dates[i+1] = dates[i] + 1d  AND  same calendar year',
    'n_pairs_total':    int(N_pairs),
    'n_pairs_train':    int(n_train),
    'n_pairs_val':      int(n_val),
    'train_years':      [int(y) for y in train_years],
    'val_years':        [int(y) for y in val_years],
    'val_year_stride':  int(VAL_YEARS_STRIDE),
    'date_min':         str(dates_t.min().date()),
    'date_max':         str(dates_t.max().date()),
    'variant_of':       'nb17 (lp25 variant; addresses synoptic-noise confound flagged in nb19 run 1)',
}
with open(f'{NSV_DATA_DIR}/nsv_data_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\nSaved 7 .npy files + nsv_data_meta.json to {NSV_DATA_DIR}')
for name in ['X_t', 'X_t1', 'dates_t', 'bsiso_phase_t', 'bsiso_amplitude_t', 'enso_cat_t', 'train_mask']:
    p = f'{NSV_DATA_DIR}/{name}.npy'
    print(f'  {name+".npy":<26s}  {os.path.getsize(p)/1e6:7.2f} MB')

## Cell 3 — Verification Gate

In [ ]:
# 1. No cross-year pairs
next_dates = dates_all[pair_idx_t1]
deltas = (next_dates - dates_t).days
assert deltas.min() == 1 and deltas.max() == 1, f'Cross-year pair detected! min={deltas.min()}, max={deltas.max()}'
assert (next_dates.year.values == dates_t.year.values).all(), 'Year mismatch in some pair!'
print(f'✓ All {N_pairs} pairs have delta = 1 day and same year.')

# 2. Pair count in reasonable range.
# nb08's lp25 preprocessing dropped 25 days from each END of EACH year's MJJAS block
# (filter half-window), so per year: 153 days - 50 = ~103 days -> ~102 pairs.
# 43 years × 102 = 4,386 typical.
expected_lo, expected_hi = 43 * 95, 43 * 105
assert expected_lo <= N_pairs <= expected_hi, f'Pair count {N_pairs} outside expected range [{expected_lo}, {expected_hi}]'
print(f'✓ Pair count {N_pairs} within expected range [{expected_lo}, {expected_hi}].')

# 3. No NaN
assert not np.isnan(X_t).any() and not np.isnan(X_t1).any(), 'NaN in X_t or X_t1!'
print(f'✓ No NaN in X_t or X_t1.')

# 4. ENSO balance
print('\nENSO category balance (anchor-day basis):')
print(f'  {"split":<6} {"El Nino":>9} {"Neutral":>9} {"La Nina":>9}  total')
for split_name, mask in [('train', train_mask), ('val', ~train_mask)]:
    cats = enso_cat_t[mask]
    n_en = int((cats == 'El Nino').sum()); n_nu = int((cats == 'Neutral').sum()); n_ln = int((cats == 'La Nina').sum())
    print(f'  {split_name:<6} {n_en:9d} {n_nu:9d} {n_ln:9d}  {mask.sum():5d}')
    assert n_en > 0 and n_nu > 0 and n_ln > 0, f'{split_name} split missing an ENSO category!'
print('✓ Both splits contain all three ENSO categories.')

# 5. BSISO phase coverage (informational)
print('\nBSISO phase distribution (anchor-day basis):')
for split_name, mask in [('train', train_mask), ('val', ~train_mask)]:
    phases = bsiso_phase_t[mask]
    counts = [int((phases == p).sum()) for p in range(1, 9)]
    print(f'  {split_name:<6} ' + ' '.join(f'P{p}={c:4d}' for p, c in zip(range(1, 9), counts)))

# 6. Per-month coverage — lp25 drops edges of May and September; June/July/August dominate
print('\nPer-month coverage (anchor-day basis):')
months_t = pd.DatetimeIndex(dates_t.values).month
for split_name, mask in [('train', train_mask), ('val', ~train_mask)]:
    counts = [int(((months_t == m) & mask).sum()) for m in [5, 6, 7, 8, 9]]
    print(f'  {split_name:<6} May={counts[0]:5d}  Jun={counts[1]:5d}  Jul={counts[2]:5d}  Aug={counts[3]:5d}  Sep={counts[4]:5d}')

# 7. Smoothness sanity vs nb17
var_per_pixel = X_t.var()
print(f'\nlp25 per-pixel variance:  {var_per_pixel:.4f}  (Lee-only nb17 should be larger; lp25 has synoptic noise removed)')

print('\n✓ Verification gate PASSED. Proceed to nb18b (Stage 1 encoder-decoder on lp25 pairs).')

## Cell 4 — Visualize Three Random Pairs (OLR channel)

Same 3-row plot as nb17 Cell 4. **Expect**: the OLR maps look noticeably **smoother** than nb17's — the fast synoptic-eddy speckle is gone, leaving only the slow intraseasonal pattern. This is the visual confirmation that the lowpass did what it was supposed to do.

In [ ]:
rng = np.random.default_rng(42)
val_pos = np.where(~train_mask)[0]
picks = rng.choice(val_pos, size=3, replace=False)

olr_ch = 2
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True, sharey=True)
fig.suptitle("lp25 — three random val pairs, OLR' anomaly map at X_t (left) and X_t+1 (right)",
             fontsize=12, fontweight='bold')

vmax = max(np.abs(X_t[picks, olr_ch]).max(), np.abs(X_t1[picks, olr_ch]).max())
for row_idx, k in enumerate(picks):
    date_t  = pd.Timestamp(dates_t[k])
    date_t1 = date_t + pd.Timedelta(days=1)
    for col_idx, (Xarr, when) in enumerate([(X_t, 't'), (X_t1, 't+1')]):
        ax = axes[row_idx, col_idx]
        im = ax.imshow(Xarr[k, olr_ch], cmap='RdBu_r', aspect='auto',
                       extent=[60, 160, 0, 60], vmin=-vmax, vmax=vmax, origin='lower')
        d = date_t if when == 't' else date_t1
        ax.set_title(f"{when}:  {d.date()}  (phase {bsiso_phase_t[k]}, ENSO {enso_cat_t[k]})",
                     fontsize=10)
        if col_idx == 0: ax.set_ylabel('Latitude (°)')
        if row_idx == 2: ax.set_xlabel('Longitude (°)')

fig.subplots_adjust(right=0.90)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
plt.colorbar(im, cax=cbar_ax, label="OLR' anomaly (σ)")
fig_path = f'{NSV_DATA_DIR}/sample_pairs_olr.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print('\nVisual check: maps should be visibly smoother than nb17 — synoptic eddies removed by lp25 lowpass.')

---
## Done!

**Send back** for review:
1. The printed output of Cell 2 — pair count, train/val sizes.
2. The printed output of Cell 3 — verification gate.
3. `sample_pairs_olr.png` — should look noticeably smoother than nb17's version (synoptic eddies removed).

**Next**: nb18b consumes `BSISO_SSL_Project/nsv/data_lp25/` and trains the same encoder/decoder as nb18, saving latents to `nsv/latents_lp25/`. Then re-run nb19 with `LATENT_DIR = NSV_DIR/latents_lp25` and `RESULTS_DIR = NSV_DIR/results/stage2_lp25` to get the corrected ID estimate.

**Expected `d̂`** on lp25 latents: 2–5, based on nb08's confirmation that the lowpassed BSISO data has clean low-D circular structure (z = 14.55 from a 2-D embedding).

---
*DDCS Project | jh9141@nyu.edu*